# Accessing the Breast Cancer Classification Tool

### Login Information
Access the platform using your **email** as username and your **student ID** as password.  
Example credentials:  
**Username:** jpeyperez@unav.es  
**Password:** 163587-6  

**Platform URL:**  
https://reto-data-analytics.intelligentbiodata.com/data-analytics/

> This is a recently developed tool. If you encounter bugs or issues, please contact: **ypey@external.unav.es**

---

### Platform Buttons

- **UPDATE LEADERBOARD**: Displays the top 10 ranked results submitted by users.
- **UPLOAD CSV FILE**: Allows you to upload a prediction file. If the format is incorrect, an error message will be shown.

---

### CSV File Requirements

To be evaluated, your CSV file must:

- Be **comma-separated** (.csv)
- Contain two required columns:
  - `"id"`: unique identifier
  - `"diagnosis"`: your prediction (malignant or benign)

> Only samples with missing values (NaN) in the `"diagnosis"` column of the file `data_breast_cancer_red.csv` will be evaluated. These represent the test set.

---

### Your Task

Given several features, your objective is to **predict whether a tumor is benign or malignant**.

- The dataset includes patients with unknown outcomes (diagnosis = NaN).
- These are the **test cases** evaluated by the web tool.
- You are free to use:
  - Any classification algorithm
  - Any set of parameters
  - Any validation strategy (train/test split, cross-validation, etc.)

---

### Evaluation and Grading

- The **leaderboard** will determine part of your final grade.
- The system will only retain and display your **best submission**.
- A maximum of **50 total submissions** is allowed per user.
- The **Jupyter notebook** used for your best submission must be uploaded to ADI.
- **Code originality** will be reviewed.
- ✅ **This task accounts for 2 points of your final grade.**

**Deadline: May 6**

In [193]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    VotingClassifier
)
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

Analyse Data

In [194]:
df = pd.read_csv('data_breast_cancer_red.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,NaN,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,NaN,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [195]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 32 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                319 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [196]:
print("Total null:", df.isnull().sum().sum())
display(df.isnull().sum())

Total null: 250


id                           0
diagnosis                  250
radius_mean                  0
texture_mean                 0
perimeter_mean               0
area_mean                    0
smoothness_mean              0
compactness_mean             0
concavity_mean               0
concave points_mean          0
symmetry_mean                0
fractal_dimension_mean       0
radius_se                    0
texture_se                   0
perimeter_se                 0
area_se                      0
smoothness_se                0
compactness_se               0
concavity_se                 0
concave points_se            0
symmetry_se                  0
fractal_dimension_se         0
radius_worst                 0
texture_worst                0
perimeter_worst              0
area_worst                   0
smoothness_worst             0
compactness_worst            0
concavity_worst              0
concave points_worst         0
symmetry_worst               0
fractal_dimension_worst      0
dtype: i

Preprocess

In [197]:
df_model = df[df['diagnosis'].notna()].copy()
df_eval = df[df['diagnosis'].isna()].copy()

X = df_model.drop(columns=['id', 'diagnosis'])
y = df_model['diagnosis'].map({'B': 0, 'M': 1})

X_eval = df_eval.drop(columns=['id', 'diagnosis'])

print('Labelled rows:', df_model.shape[0])
print('Unlabelled rows:', df_eval.shape[0])
print('Features:', X.shape[1])

Labelled rows: 319
Unlabelled rows: 250
Features: 30


Split Data

In [198]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Scale features for models that need it
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns, index=X.index)
X_eval_scaled = pd.DataFrame(scaler.transform(X_eval), columns=X_eval.columns, index=X_eval.index)

# Define individual models
xgb = XGBClassifier(
    n_estimators=500, max_depth=2, learning_rate=0.02,
    subsample=0.85, colsample_bytree=0.85,
    min_child_weight=2, gamma=0.1,
    random_state=42, eval_metric='logloss'
)

rf = RandomForestClassifier(
    n_estimators=500, max_depth=None,
    min_samples_split=5, min_samples_leaf=2,
    random_state=42, class_weight='balanced'
)

svm = SVC(
    C=1.0, kernel='rbf', gamma='scale',
    probability=True, random_state=42
)

lr = LogisticRegression(
    C=0.5, max_iter=5000,
    class_weight='balanced', random_state=42
)

gb = GradientBoostingClassifier(
    n_estimators=300, max_depth=2, learning_rate=0.03,
    subsample=0.85, random_state=42
)

# Soft voting ensemble (uses predicted probabilities)
ensemble = VotingClassifier(
    estimators=[
        ('xgb', xgb),
        ('rf', rf),
        ('svm', svm),
        ('lr', lr),
        ('gb', gb)
    ],
    voting='soft'
)

# Cross-validate each model and the ensemble
print('--- Individual model CV scores ---')
for name, mdl in [('XGBoost', xgb), ('RandomForest', rf), ('SVM', svm),
                   ('LogReg', lr), ('GradBoost', gb), ('Ensemble', ensemble)]:
    s = cross_val_score(mdl, X_scaled, y, cv=cv, scoring='accuracy')
    print(f'{name:>12}: {s.mean():.4f}  {s}')

--- Individual model CV scores ---
     XGBoost: 0.9530  [0.953125   0.953125   0.953125   0.953125   0.95238095]
RandomForest: 0.9530  [0.953125   0.9375     0.953125   0.96875    0.95238095]
         SVM: 0.9655  [0.96875    0.96875    0.984375   0.9375     0.96825397]
      LogReg: 0.9717  [0.96875    0.984375   0.984375   0.96875    0.95238095]
   GradBoost: 0.9531  [0.96875    0.9375     0.953125   0.921875   0.98412698]
    Ensemble: 0.9656  [0.96875    0.953125   0.96875    0.953125   0.98412698]


In [199]:
X_tr, X_vl, y_tr, y_vl = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42, stratify=y
)

ensemble.fit(X_tr, y_tr)
y_vl_pred = ensemble.predict(X_vl)

print('Validation accuracy:', accuracy_score(y_vl, y_vl_pred))
print()
print('Confusion matrix:')
print(confusion_matrix(y_vl, y_vl_pred))
print()
print(classification_report(y_vl, y_vl_pred, target_names=['B (0)', 'M (1)']))

Validation accuracy: 0.975

Confusion matrix:
[[49  1]
 [ 1 29]]

              precision    recall  f1-score   support

       B (0)       0.98      0.98      0.98        50
       M (1)       0.97      0.97      0.97        30

    accuracy                           0.97        80
   macro avg       0.97      0.97      0.97        80
weighted avg       0.97      0.97      0.97        80



In [200]:
ensemble.fit(X_scaled, y)

VotingClassifier(estimators=[('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=0.85, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric='logloss',
                                            feature_types=None,
                                            feature_weights=None, gamma=0.1,
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_con...
                              RandomForestClassifier(class_weight='balanced',
                                                     min_samples_leaf=2,
                                                     min_samples_split=5,
                                                     n_estimators=500,
                                                     random_state=42)),
                             ('svm', SVC(probability=True, random_state=42)),
                             ('lr',
                              LogisticRegression(C=0.5, class_weight='balanced',
                                                 max_iter=5000,
                                                 random_state=42)),
                             ('gb',
                              GradientBoostingClassifier(learning_rate=0.03,
                                                         max_depth=2,
                                                         n_estimators=300,
                                                         random_state=42,
                                                         subsample=0.85))],
                 voting='soft')

In [201]:
preds = ensemble.predict(X_eval_scaled)
preds = ['M' if p == 1 else 'B' for p in preds]

submission = pd.DataFrame({
    'id': df_eval['id'],
    'diagnosis': preds
})

submission.to_csv('submission.csv', index=False)
print(submission['diagnosis'].value_counts())
submission.head()

diagnosis
B    159
M     91
Name: count, dtype: int64


,id,diagnosis
2,84300903,M
4,84358402,M
8,844981,M
9,84501001,M
10,845636,M


In [202]:
# Old experimental model cell intentionally disabled.
# The final model above is trained on all labelled rows before creating submission.csv.

In [203]:
# Old cross-validation cell intentionally disabled.
# Cross-validation is already computed above with StratifiedKFold.

In [204]:
# Old submission cell intentionally disabled.
# The correct submission uses df_eval['id'], not df_eval.index.x